In [40]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import Lasso
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_predict, KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

## Load preprocessed data

In [41]:
df = pd.read_csv("../data/processed/combined_features.csv", index_col='ticker')

X = df.drop(columns=['ev_to_ebitda'])
y = df['ev_to_ebitda']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (2271, 391), Test: (568, 391)


In [42]:
# Diagnose and clean inf/extreme values before modeling
print("=== Inf/NaN check before cleaning ===")
print("Inf in X:", np.isinf(X).sum().sum())
print("NaN in X:", X.isna().sum().sum())
print("Inf in y:", np.isinf(y).sum())
print("NaN in y:", y.isna().sum())
print("\nInf counts per column:\n", np.isinf(X).sum()[np.isinf(X).sum() > 0])

# Replace inf/-inf with NaN, then drop rows with any NaN in X or y
X = X.replace([np.inf, -np.inf], np.nan)
y = y.replace([np.inf, -np.inf], np.nan)
mask = X.notna().all(axis=1) & y.notna()
X, y = X[mask], y[mask]

print(f"\nAfter cleaning: {X.shape[0]} rows remaining (dropped {(~mask).sum()})")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

=== Inf/NaN check before cleaning ===
Inf in X: 3
NaN in X: 0
Inf in y: 0
NaN in y: 0

Inf counts per column:
 forwardPE    3
dtype: int64

After cleaning: 2836 rows remaining (dropped 3)
Train: (2268, 391), Test: (568, 391)


In [ ]:
# Filter target to realistic range, log-transform scale-dependent features
print(f"y stats before filter: min={y.min():.1f}, max={y.max():.1f}, median={y.median():.1f}")

before = len(X)
mask = (y > 0) & (y < 50)
X = X[mask].copy()
y = y[mask].copy()
print(f"Target filter (0 < ev_to_ebitda < 50): kept {len(X)}/{before} rows")

# Log-transform scale-dependent features (heavy right tails across mega-caps vs small-caps)
log_cols = ['shares_outstanding', 'total_debt', 'total_cash', 'ebitda']
for c in log_cols:
    # signed log1p handles negatives (e.g., negative ebitda for unprofitable firms)
    X[c] = np.sign(X[c]) * np.log1p(np.abs(X[c]))

# Log-transform the target — predict log(ev/ebitda) instead of the raw ratio
y = np.log1p(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"y range after log1p: [{y.min():.2f}, {y.max():.2f}]")

## Define feature groups

In [43]:
quant_columns = ['shares_outstanding', 'total_debt', 'total_cash', 'ebitda',
                 'debt_to_equity', 'ebitda_margin', 'forwardPE']
embed_columns = [col for col in X_train.columns if col.startswith('embed_')]

print(f"Quant: {len(quant_columns)}, Embed: {len(embed_columns)}")

Quant: 7, Embed: 384


## Extract and scale features

In [44]:
scaler = StandardScaler()
X_quant_train = scaler.fit_transform(X_train[quant_columns])
X_quant_test  = scaler.transform(X_test[quant_columns])

X_embed_train = X_train[embed_columns].values
X_embed_test  = X_test[embed_columns].values

X_all_train = np.column_stack((X_quant_train, X_embed_train))
X_all_test  = np.column_stack((X_quant_test,  X_embed_test))

## Plan A: Stacking Ensemble (quant + embed → meta-learner)

In [45]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

model_quant = HistGradientBoostingRegressor(random_state=42)
model_embed = KNeighborsRegressor(n_neighbors=5, metric='cosine', weights='distance')

## Generate meta-features via cross-val predict

In [46]:
pred_quant = cross_val_predict(model_quant, X_quant_train, y_train, cv=cv)
pred_embed = cross_val_predict(model_embed, X_embed_train, y_train, cv=cv)

X_meta_train = np.column_stack((pred_quant, pred_embed))

print('--- CV R2 per Expert ---')
print(f"Quant: {r2_score(y_train, pred_quant):.4f}")
print(f"Embed: {r2_score(y_train, pred_embed):.4f}")

--- CV R2 per Expert ---
Quant: 0.0327
Embed: -0.1200


## Fit experts on full train set, generate test meta-features

In [47]:
model_quant.fit(X_quant_train, y_train)
model_embed.fit(X_embed_train, y_train)

X_meta_test = np.column_stack([
    model_quant.predict(X_quant_test),
    model_embed.predict(X_embed_test),
])

## Fit meta-learner and evaluate

In [48]:
meta = Lasso(alpha=0.5, positive=True)
meta.fit(X_meta_train, y_train)
y_pred_a = meta.predict(X_meta_test)

print("=== Plan A Results ===")
print(f"Test R2:  {r2_score(y_test, y_pred_a):.4f}")
print(f"Test MSE: {mean_squared_error(y_test, y_pred_a):.4f}")
print(f"Weights — quant: {meta.coef_[0]:.3f}, embed: {meta.coef_[1]:.3f}")

=== Plan A Results ===
Test R2:  0.0145
Test MSE: 258922.0110
Weights — quant: 0.585, embed: 0.003


## Plan B: XGBoost on all features

In [49]:
model_xgb = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4,
                              subsample=0.8, colsample_bytree=0.8,
                              random_state=42, verbosity=0)

model_xgb.fit(X_all_train, y_train)
y_pred_b = model_xgb.predict(X_all_test)

print("=== Plan B Results ===")
print(f"Test R2:  {r2_score(y_test, y_pred_b):.4f}")
print(f"Test MSE: {mean_squared_error(y_test, y_pred_b):.4f}")

=== Plan B Results ===
Test R2:  -0.0018
Test MSE: 263201.0682
